# NB1 — Pipeline de análisis estadístico

Reproduce de forma transparente todas las tablas y cifras de `informe/11-RESULTADOS.md`.

**Fuentes de datos:**
- `<RUNS_DIR>/produccion/{tier}/{scenario}/{arm}/{deadlock_strategy}/seed_N.summary.json`
- `<MISSIONS_DIR>/{scenario}.json` — manifiestos de misión para SPL

**Cómo ejecutar:**
1. Activar el conda env: `conda activate airsimenv`
2. Ajustar `RUNS_DIR` y `MISSIONS_DIR` si los datos están en otra ubicación
3. Kernel → Run All Cells

Las tablas y figuras se exportan a `output/` y `output/figures/`.

## §0 — Configuración

In [ ]:
import glob
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import mannwhitneyu

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Directorios — ajustar si los datos están en otra ubicación
RUNS_DIR      = os.environ.get('RUNS_DIR',      str(Path('../airsim-runs/produccion').resolve()))
MISSIONS_DIR  = os.environ.get('MISSIONS_DIR',  str(Path('../airsim-plan/missions/flightplans').resolve()))
OUTPUT_DIR    = Path('notebooks/output')
FIGURES_DIR   = OUTPUT_DIR / 'figures'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Escenarios del lote base (45 corridas con success=True)
LOTE_BASE = ['minisim_clear', 'townsim_clear', 'citysim_clear']
ARM_ORDER = ['slm', 'fsm', 'reactive']
TIER_LABELS = {
    'minisim_clear': 'Tier 0 (minisim_clear)',
    'townsim_clear': 'Tier 1 (townsim_clear)',
    'citysim_clear': 'Tier 2 (citysim_clear)',
    'townsim_ini':   'Tier 1 (townsim_ini)',
}

print(f'RUNS_DIR:     {RUNS_DIR}')
print(f'MISSIONS_DIR: {MISSIONS_DIR}')

## §1 — Carga de datos

In [ ]:
def load_summaries(runs_dir: str) -> pd.DataFrame:
    """Lee todos los summary.json recursivamente y devuelve un DataFrame con una fila por corrida."""
    rows = []
    for path in glob.glob(str(Path(runs_dir) / '**' / '*.summary.json'), recursive=True):
        try:
            with open(path, encoding='utf-8') as f:
                d = json.load(f)
            d['_path'] = path
            rows.append(d)
        except Exception as e:
            print(f'  WARN: {path}: {e}')
    df = pd.DataFrame(rows)
    # Normalizar nombres de columna opcionales
    for col in ['code_version', 'deadlock_strategy', 'deadlock_events',
                'deep_scan_events', 'deep_scan_resolution_rate',
                'deep_scan_avg_cycles_to_resolve', 'deep_scan_fallback_rate']:
        if col not in df.columns:
            df[col] = None
    return df


df_all = load_summaries(RUNS_DIR)
print(f'Corridas cargadas: {len(df_all)}')
print(df_all.dtypes)
df_all.head(10)

In [ ]:
def load_optimal_lengths(missions_dir: str) -> dict:
    """Longitud óptima por escenario: suma de distancias entre waypoints consecutivos del manifiesto."""
    lengths = {}
    for path in glob.glob(str(Path(missions_dir) / '*.json')):
        try:
            with open(path, encoding='utf-8') as f:
                manifest = json.load(f)
        except Exception:
            continue
        wps = manifest.get('waypoints', [])
        total = 0.0
        for i in range(1, len(wps)):
            a, b = wps[i-1], wps[i]
            total += math.dist(
                (a.get('x', 0.), a.get('y', 0.), a.get('z', 0.)),
                (b.get('x', 0.), b.get('y', 0.), b.get('z', 0.))
            )
        stem = Path(path).stem
        lengths[stem] = total
    return lengths


L_opt = load_optimal_lengths(MISSIONS_DIR)
for sc in LOTE_BASE + ['townsim_ini']:
    print(f'  {sc}: L_opt = {L_opt.get(sc, "N/A"):.1f} m')

In [ ]:
def compute_spl(row, optimal_lengths: dict) -> float:
    """SPL = L_opt / max(L_recorrida, L_opt) si success, 0 si falla, NaN si no hay manifiesto."""
    l_opt = optimal_lengths.get(row['scenario'])
    if l_opt is None:
        return float('nan')
    if not row.get('success'):
        return 0.0
    l_actual = max(float(row.get('path_length_m', 0.) or 0.), 1e-6)
    return l_opt / max(l_actual, l_opt)


df_all['spl'] = df_all.apply(compute_spl, axis=1, optimal_lengths=L_opt)

# Lote base: sólo los tres escenarios con success=True
df_base = df_all[df_all['scenario'].isin(LOTE_BASE)].copy()
print(f'Corridas del lote base: {len(df_base)} (éxitos: {df_base["success"].sum()})')
df_base[['scenario', 'arm', 'seed', 'duration_s', 'path_length_m', 'slm_invocations',
          'deliberation_rate', 'deadlock_events', 'success']].head(15)

## §2 — Tabla §11.1: resultados por tier (media ± σ)

Reproduce las tablas Tier 0 / Tier 1 / Tier 2 de §11.1 del informe.

In [ ]:
def tier_table(df: pd.DataFrame, scenario: str) -> pd.DataFrame:
    """Genera la sub-tabla de §11.1 para un escenario dado."""
    sub = df[df['scenario'] == scenario]
    rows = []
    for arm in ARM_ORDER:
        g = sub[sub['arm'] == arm]
        if g.empty:
            continue
        success = f"{int(g['success'].sum())}/{len(g)} {'✅' if g['success'].all() else '⛔'}"
        dur_mean = g['duration_s'].mean()
        dur_std  = g['duration_s'].std(ddof=1)
        dist_mean= g['path_length_m'].mean()
        dist_std = g['path_length_m'].std(ddof=1)
        col_km   = g['collisions'].sum() / (g['path_length_m'].sum() / 1000) if g['path_length_m'].sum() > 0 else 0
        slm_inv  = g['slm_invocations'].mean() if arm == 'slm' else float('nan')
        delib    = g['deliberation_rate'].mean() * 100 if arm == 'slm' else float('nan')
        fallback = g['slm_fallback_rate'].mean() * 100 if arm == 'slm' else float('nan')
        dlock    = g['deadlock_events'].mean() if 'deadlock_events' in g.columns else float('nan')
        res_rate = g['deep_scan_resolution_rate'].mean() * 100 if 'deep_scan_resolution_rate' in g.columns else float('nan')
        dist_min = g['min_obstacle_dist_m'].mean()
        rows.append({
            'Brazo': arm,
            'Éxito': success,
            'Duración media (s)': f'{dur_mean:.1f}',
            'σ (s)': f'{dur_std:.1f}',
            'Dist. media (m)': f'{dist_mean:.1f}',
            'σ (m)': f'{dist_std:.1f}',
            'Col./km': f'{col_km:.2f}',
            'Invoc. SLM': f'{slm_inv:.1f}' if not math.isnan(slm_inv) else '—',
            'Deliberación': f'{delib:.1f}%' if not math.isnan(delib) else '—',
            'Fallback SLM': f'{fallback:.1f}%' if not math.isnan(fallback) else '—',
            'Deadlocks': f'{dlock:.1f}' if not math.isnan(dlock) else '—',
            'Res. atasco VLM': f'{res_rate:.0f}%' if not math.isnan(res_rate) else '—',
            'DistMin (m)': f'{dist_min:.1f}' if not math.isnan(dist_min) else '—',
        })
    return pd.DataFrame(rows).set_index('Brazo')


for sc in LOTE_BASE:
    print(f'\n=== {TIER_LABELS[sc]} ===')
    t = tier_table(df_base, sc)
    display(t)
    t.to_csv(OUTPUT_DIR / f'tabla_11_1_{sc}.csv')

In [ ]:
# Verificación: comparar con valores de 11-RESULTADOS.md (tolerancia ±0.5 s)
EXPECTED = {
    ('slm',      'minisim_clear'): 164.2,
    ('fsm',      'minisim_clear'): 174.2,
    ('reactive', 'minisim_clear'):  74.7,
    ('slm',      'townsim_clear'): 296.0,
    ('fsm',      'townsim_clear'): 329.5,
    ('reactive', 'townsim_clear'): 259.7,
    ('slm',      'citysim_clear'): 175.5,
    ('fsm',      'citysim_clear'): 187.7,
    ('reactive', 'citysim_clear'): 165.2,
}
TOL = 0.5
ok_count = 0
for (arm, sc), expected_val in EXPECTED.items():
    g = df_base[(df_base['arm'] == arm) & (df_base['scenario'] == sc)]
    computed = g['duration_s'].mean()
    delta = abs(computed - expected_val)
    status = '✅' if delta <= TOL else f'⚠️  Δ={delta:.2f}'
    print(f'{status}  {arm:<10} {sc:<18}  esperado={expected_val:.1f}  calculado={computed:.1f}')
    if delta <= TOL:
        ok_count += 1
print(f'\n{ok_count}/{len(EXPECTED)} valores dentro de ±{TOL}s')

## §3 — Tabla §11.2: resumen cruzado por brazo y escenario

In [ ]:
def cross_table(df: pd.DataFrame) -> pd.DataFrame:
    """Tabla §11.2 — una fila por (brazo, escenario) con ratio vs reactive."""
    # Tiempos medios del brazo reactive por escenario (denominador del ratio)
    reactive_mean = {}
    for sc in df['scenario'].unique():
        r = df[(df['arm'] == 'reactive') & (df['scenario'] == sc)]
        if not r.empty:
            reactive_mean[sc] = r['duration_s'].mean()

    rows = []
    scenarios_ordered = LOTE_BASE + [s for s in df['scenario'].unique() if s not in LOTE_BASE]
    for arm in ARM_ORDER:
        for sc in scenarios_ordered:
            g = df[(df['arm'] == arm) & (df['scenario'] == sc)]
            if g.empty:
                continue
            dur_mean = g['duration_s'].mean()
            dur_std  = g['duration_s'].std(ddof=1)
            success_str = f"{int(g['success'].sum())}/{len(g)}"
            col_km = g['collisions'].sum() / (g['path_length_m'].sum() / 1000) if g['path_length_m'].sum() > 0 else 0
            dist_min = g['min_obstacle_dist_m'].mean()
            delib = g['deliberation_rate'].mean() * 100 if arm == 'slm' else float('nan')
            fallback = g['slm_fallback_rate'].mean() * 100 if arm == 'slm' else float('nan')
            dlock = g['deadlock_events'].mean()
            res_rate = g['deep_scan_resolution_rate'].mean() * 100
            ratio = dur_mean / reactive_mean[sc] if sc in reactive_mean else float('nan')
            rows.append({
                'Brazo': arm,
                'Tier / Escenario': TIER_LABELS.get(sc, sc),
                'Éxito': success_str,
                'Col./km': f'{col_km:.0f}',
                'DistMin (m)': f'{dist_min:.1f}' if not math.isnan(dist_min) else '—',
                'Tiempo (s)': f'{dur_mean:.1f} ± {dur_std:.1f}',
                'Deliberación': f'{delib:.1f}%' if not math.isnan(delib) else '—',
                'Fallback SLM': f'{fallback:.1f}%' if not math.isnan(fallback) else '—',
                'Res. Atasco VLM': f'{res_rate:.0f}% ({dlock:.1f}/c.)' if not math.isnan(dlock) and dlock > 0 else '— (0 deadlocks)',
                'Ratio vs reactive': f'{ratio:.2f}×' if not math.isnan(ratio) else '—',
            })
    return pd.DataFrame(rows)


df_cross = cross_table(df_all[df_all['scenario'].isin(LOTE_BASE + ['townsim_ini'])])
display(df_cross)
df_cross.to_csv(OUTPUT_DIR / 'tabla_11_2_cross.csv', index=False)

## §4 — Tabla §11.3.1: ratios de tiempo de misión (H2)

In [ ]:
means = df_base.groupby(['scenario', 'arm'])['duration_s'].mean().unstack()
means = means.reindex(columns=ARM_ORDER)

ratios = pd.DataFrame(index=LOTE_BASE)
ratios['Tiempo reactive (s)'] = means.loc[LOTE_BASE, 'reactive'].map(lambda v: f'{v:.1f}')
ratios['Tiempo slm (s)'] = means.loc[LOTE_BASE, 'slm'].map(lambda v: f'{v:.1f}')
ratios['Ratio slm/reactive'] = (means.loc[LOTE_BASE, 'slm'] / means.loc[LOTE_BASE, 'reactive']).map(lambda v: f'{v:.2f}×')
ratios['Tiempo fsm (s)'] = means.loc[LOTE_BASE, 'fsm'].map(lambda v: f'{v:.1f}')
ratios['Ratio fsm/reactive'] = (means.loc[LOTE_BASE, 'fsm'] / means.loc[LOTE_BASE, 'reactive']).map(lambda v: f'{v:.2f}×')
inv_mean = df_base[df_base['arm'] == 'slm'].groupby('scenario')['slm_invocations'].mean()
ratios['Invoc. VLM (slm)'] = inv_mean.loc[LOTE_BASE].map(lambda v: f'{v:.1f}')
ratios.index = [TIER_LABELS[s] for s in LOTE_BASE]

display(ratios)
ratios.to_csv(OUTPUT_DIR / 'tabla_11_3_1_ratios.csv')

In [ ]:
# Gráfico: ratio vs. longitud de ruta (muestra la dilución del overhead)
route_lengths = {'minisim_clear': 184., 'townsim_clear': 626., 'citysim_clear': 431.}

fig, ax = plt.subplots(figsize=(7, 4))
for arm, color, marker in [('slm', '#2196F3', 'o'), ('fsm', '#FF9800', 's')]:
    x = [route_lengths[sc] for sc in LOTE_BASE]
    y_mean = (means.loc[LOTE_BASE, arm] / means.loc[LOTE_BASE, 'reactive']).values
    ax.plot(x, y_mean, marker=marker, color=color, label=arm, linewidth=2, markersize=8)
    for xi, yi, sc in zip(x, y_mean, LOTE_BASE):
        ax.annotate(f'{yi:.2f}×', (xi, yi), textcoords='offset points', xytext=(6, 4), fontsize=9, color=color)

ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, label='reactive (1.00×)')
ax.set_xlabel('Longitud de ruta (m)')
ax.set_ylabel('Ratio tiempo / tiempo reactive')
ax.set_title('Dilución del overhead deliberativo con la longitud de ruta')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_ratios_vs_ruta.png', dpi=150)
plt.show()

## §5 — Prueba Mann-Whitney U + Cliff's δ (§11.3.2)

Métrica: `duration_s`. Prueba bilateral. Corrección de Bonferroni: 18 comparaciones (3 pares × 6 escenarios incluyendo extendidos).

**Nota:** con K=5 semillas, el p mínimo alcanzable es 7.9×10⁻³ — por diseño no puede superar Bonferroni con α_ajustado = 0.0028.

In [ ]:
def cliffs_delta(x, y) -> float:
    """Cliff's delta: fracción de pares donde x > y menos fracción donde x < y."""
    x, y = list(x), list(y)
    n = len(x) * len(y)
    if n == 0:
        return float('nan')
    concordant = sum(1 if xi > yj else -1 if xi < yj else 0 for xi in x for yj in y)
    return concordant / n


def effect_label(delta: float) -> str:
    """Etiqueta del tamaño de efecto según Romano et al. 2006."""
    d = abs(delta)
    if d < 0.147: return 'negligible'
    if d < 0.330: return 'pequeño'
    if d < 0.474: return 'mediano'
    return 'grande'


# Pares de comparación
PAIRS = [('slm', 'reactive'), ('fsm', 'reactive'), ('slm', 'fsm')]
N_COMPARACIONES = 18
ALPHA = 0.05
ALPHA_BONF = ALPHA / N_COMPARACIONES

mwu_rows = []
for sc in LOTE_BASE:
    for arm_a, arm_b in PAIRS:
        a = df_base[(df_base['arm'] == arm_a) & (df_base['scenario'] == sc)]['duration_s'].values
        b = df_base[(df_base['arm'] == arm_b) & (df_base['scenario'] == sc)]['duration_s'].values
        if len(a) < 2 or len(b) < 2:
            continue
        U, p = mannwhitneyu(a, b, alternative='two-sided')
        n1, n2 = len(a), len(b)
        # Estadístico z normal approximation
        mean_U = n1 * n2 / 2
        std_U = math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
        z = (U - mean_U) / std_U if std_U > 0 else 0
        p_bonf = min(p * N_COMPARACIONES, 1.0)
        delta = cliffs_delta(a, b)
        sig = '**' if p_bonf < ALPHA else ('*' if p < ALPHA else 'ns')
        sig_str = f'{sig} (no Bonf.)' if p < ALPHA and p_bonf >= ALPHA else sig
        mwu_rows.append({
            'Comparación': f'`{arm_a}` vs `{arm_b}`',
            'Escenario': TIER_LABELS[sc],
            'U': f'{U:.0f}',
            'z': f'{z:.2f}',
            'p': f'{p:.3f}',
            'p (Bonf.)': f'{p_bonf:.3f}',
            "Cliff's δ": f'{delta:+.2f}',
            'Efecto': effect_label(delta),
            'Significancia': sig_str,
        })

df_mwu = pd.DataFrame(mwu_rows)
display(df_mwu)
df_mwu.to_csv(OUTPUT_DIR / 'tabla_11_3_2_mwu.csv', index=False)
print(f'\nα_Bonferroni = {ALPHA}/{N_COMPARACIONES} = {ALPHA_BONF:.4f}')

In [ ]:
# Boxplot: distribución de tiempos por brazo y escenario
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
palette = {'slm': '#2196F3', 'fsm': '#FF9800', 'reactive': '#4CAF50'}

for ax, sc in zip(axes, LOTE_BASE):
    sub = df_base[df_base['scenario'] == sc]
    sns.boxplot(data=sub, x='arm', y='duration_s', order=ARM_ORDER, palette=palette, ax=ax,
                width=0.5, flierprops=dict(marker='x', markeredgecolor='gray'))
    sns.stripplot(data=sub, x='arm', y='duration_s', order=ARM_ORDER, color='black', size=4,
                  jitter=True, ax=ax, alpha=0.6)
    ax.set_title(TIER_LABELS[sc], fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('Duración (s)')

fig.suptitle('Distribución de tiempos de misión por brazo — lote base (5 semillas)', fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_boxplot_duration.png', dpi=150)
plt.show()

## §6 — Perfil de deliberación por tier (§11.3.3)

In [ ]:
slm_base = df_base[df_base['arm'] == 'slm'].copy()
slm_base['dist_per_invoc_m'] = slm_base['path_length_m'] / slm_base['slm_invocations'].clip(lower=1)

delib_table = slm_base.groupby('scenario').agg(
    invoc_mean=('slm_invocations', 'mean'),
    delib_rate=('deliberation_rate', 'mean'),
    dist_per_invoc=('dist_per_invoc_m', 'mean'),
).reindex(LOTE_BASE)

delib_display = pd.DataFrame({
    'Tier': [TIER_LABELS[s] for s in LOTE_BASE],
    'Invoc. VLM promedio': delib_table['invoc_mean'].map('{:.1f}'.format),
    'Tasa deliberación': (delib_table['delib_rate'] * 100).map('{:.1f}%'.format),
    'Dist. / invocación (m)': delib_table['dist_per_invoc'].map('{:.1f}'.format),
})
display(delib_display)
delib_display.to_csv(OUTPUT_DIR / 'tabla_11_3_3_deliberacion.csv', index=False)

In [ ]:
# Gráfico: invocaciones y dist/invocación vs. longitud de ruta
route_len = {'minisim_clear': 184., 'townsim_clear': 626., 'citysim_clear': 431.}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

x_pts = [route_len[sc] for sc in LOTE_BASE]
inv_pts = [delib_table.loc[sc, 'invoc_mean'] for sc in LOTE_BASE]
dp_pts  = [delib_table.loc[sc, 'dist_per_invoc'] for sc in LOTE_BASE]

ax1.plot(x_pts, inv_pts, 'o-', color='#2196F3', linewidth=2, markersize=8)
for xi, yi, sc in zip(x_pts, inv_pts, LOTE_BASE):
    ax1.annotate(f'{yi:.1f}', (xi, yi), textcoords='offset points', xytext=(6, 4))
ax1.set_xlabel('Longitud de ruta (m)')
ax1.set_ylabel('Invocaciones VLM promedio')
ax1.set_title('Invocaciones vs. longitud de ruta')

ax2.plot(x_pts, dp_pts, 's-', color='#FF5722', linewidth=2, markersize=8)
for xi, yi, sc in zip(x_pts, dp_pts, LOTE_BASE):
    ax2.annotate(f'{yi:.0f} m', (xi, yi), textcoords='offset points', xytext=(6, 4))
ax2.set_xlabel('Longitud de ruta (m)')
ax2.set_ylabel('Distancia por invocación (m/invoc.)')
ax2.set_title('Distancia por invocación vs. longitud de ruta')

fig.suptitle('Perfil de deliberación del brazo SLM — lote base', fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_deliberacion_vs_ruta.png', dpi=150)
plt.show()

## §7 — Latencia por ciclo (de archivos JSONL)

Lee los archivos `.jsonl` de ciclos para obtener la latencia del grafo por rama de control.
Esta sección puede tardar varios minutos en runs con muchos ciclos.

In [ ]:
def load_cycles(runs_dir: str, scenarios=None) -> pd.DataFrame:
    """Lee los JSONL del lote base y extrae latencia del grafo por ciclo."""
    records = []
    pattern = str(Path(runs_dir) / '**' / '*.jsonl')
    for path in glob.glob(pattern, recursive=True):
        if path.endswith('.summary.json'):
            continue
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                sc = rec.get('scenario', '')
                if scenarios and sc not in scenarios:
                    continue
                lat = rec.get('latency_ms', {}) or {}
                records.append({
                    'arm': rec.get('arm'),
                    'scenario': sc,
                    'seed': rec.get('seed'),
                    'route': rec.get('route'),
                    'latency_graph_ms': lat.get('graph', float('nan')),
                })
    return pd.DataFrame(records)


print('Cargando ciclos JSONL (puede tardar ~30-60s)...')
df_cycles = load_cycles(RUNS_DIR, scenarios=LOTE_BASE)
print(f'Ciclos cargados: {len(df_cycles):,}')
df_cycles.head()

In [ ]:
# p50 / p95 por (arm, route)
lat_stats = df_cycles.groupby(['arm', 'route'])['latency_graph_ms'].agg(
    N='count',
    p50=lambda x: x.quantile(0.50),
    p95=lambda x: x.quantile(0.95),
).reset_index().sort_values(['arm', 'route'])
display(lat_stats)
lat_stats.to_csv(OUTPUT_DIR / 'tabla_latencias_ciclo.csv', index=False)

In [ ]:
# Boxplot de latencias por ruta (agrupado por arm)
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, arm in zip(axes, ARM_ORDER):
    sub = df_cycles[df_cycles['arm'] == arm]
    routes = sub['route'].dropna().unique()
    sns.boxplot(data=sub, x='route', y='latency_graph_ms', ax=ax,
                order=sorted(routes), showfliers=False, palette='Set2')
    ax.set_title(f'Brazo: {arm}')
    ax.set_xlabel('Ruta activa')
    ax.set_ylabel('Latencia grafo (ms)')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Latencia del grafo por rama de control (p50/IQR)', fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_latencia_por_ruta.png', dpi=150)
plt.show()

## §8 — Escenario de falla: `townsim_ini` (§11.4.2b)

Escenario de obstáculo real al inicio: los tres brazos fallan (0 éxitos). Métrica relevante: distancia recorrida antes del timeout (900s).

In [ ]:
df_ini = df_all[df_all['scenario'] == 'townsim_ini'].copy()
if df_ini.empty:
    print('No se encontraron corridas de townsim_ini en RUNS_DIR')
else:
    print(f'Corridas townsim_ini: {len(df_ini)} | Éxitos: {df_ini["success"].sum()}')

    # Tabla: media ± σ de distancia recorrida, deadlocks, DistMin por brazo
    ini_stats = df_ini.groupby('arm').agg(
        N=('duration_s', 'count'),
        dur_mean=('duration_s', 'mean'),
        dur_std=('duration_s', 'std'),
        dist_mean=('path_length_m', 'mean'),
        dist_std=('path_length_m', 'std'),
        dlock_mean=('deadlock_events', 'mean'),
        dist_min=('min_obstacle_dist_m', 'mean'),
    ).reindex(ARM_ORDER).dropna(how='all')
    display(ini_stats)
    ini_stats.to_csv(OUTPUT_DIR / 'tabla_townsim_ini.csv')

    # Boxplot de distancia recorrida por brazo
    fig, ax = plt.subplots(figsize=(6, 4))
    arms_present = [a for a in ARM_ORDER if a in df_ini['arm'].unique()]
    sns.boxplot(data=df_ini, x='arm', y='path_length_m', order=arms_present,
                palette={'slm': '#2196F3', 'fsm': '#FF9800', 'reactive': '#4CAF50'}, ax=ax)
    sns.stripplot(data=df_ini, x='arm', y='path_length_m', order=arms_present,
                  color='black', size=5, jitter=True, ax=ax, alpha=0.7)
    ax.set_title('Distancia recorrida antes del timeout — townsim_ini (0 éxitos)')
    ax.set_xlabel('Brazo')
    ax.set_ylabel('Distancia recorrida (m)')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'fig_townsim_ini_dist.png', dpi=150)
    plt.show()

## §9 — Exportación

Todas las tablas se guardan en `output/*.csv` y las figuras en `output/figures/*.png`.

In [ ]:
# Confirmar archivos generados
print('Tablas CSV:')
for p in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f'  {p.name}')

print('\nFiguras PNG:')
for p in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {p.name}')